# 04 — From NetCDF to Tensogram (with xarray)

<sup>(C) Copyright 2026- ECMWF and individual contributors. Licensed under the Apache Licence, Version 2.0.</sup>

**What you will learn**

- Convert a CF-compliant NetCDF file to Tensogram with `tensogram.convert_netcdf(...)`.
- Choose between `split_by="file"` (one Tensogram message per input file) and `split_by="variable"` (one per variable).
- Lift CF attributes into `base[i]["cf"]` with `cf=True`.
- Open the resulting `.tgm` file as an xarray Dataset using `engine="tensogram"`.
- Cross-verify: decode the Tensogram result vs opening the source NetCDF with xarray directly — they should match bit-for-bit.

**Prerequisites**

- tensogram Python bindings built with `--features netcdf` — see [`README.md`](README.md).
- `libnetcdf` + `libhdf5` installed at the OS level.
- `tensogram[xarray]` installed (pulled in by `pip install -e examples/jupyter`).

> **Note on the demo NetCDF.** This notebook builds its own small NetCDF file in-memory using `netCDF4` so the cell is fully self-contained. **In a real application you'd skip this step and point at your existing `.nc` file on disk.**

In [ ]:
import matplotlib

matplotlib.use("Agg")

import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import tensogram

if not getattr(tensogram, "__has_netcdf__", False):
    print(
        "ERROR: tensogram was built without the 'netcdf' feature.\n"
        "Rebuild with: cd python/bindings && maturin develop --features netcdf",
        file=sys.stderr,
    )
    sys.exit(0)

try:
    import netCDF4
except ImportError:
    print(
        "ERROR: netCDF4 not installed. Install with: uv pip install netCDF4",
        file=sys.stderr,
    )
    sys.exit(0)

try:
    import xarray as xr
    import tensogram_xarray  # side-effect: registers engine="tensogram"
    _HAS_XARRAY = True
except ImportError:
    _HAS_XARRAY = False
    print("(xarray backend not installed — section 6 will be skipped)")

## 1. Build a demo NetCDF — *not* something you'd do in real code

The cell below builds a small CF-1.10 compliant NetCDF file with a 3D temperature variable (`time × lat × lon`) plus coordinate variables. This is only here so the notebook is self-contained; in a real workflow you'd skip ahead to section 2 with your existing `.nc` file.

In [ ]:
tmpdir = Path(tempfile.mkdtemp(prefix="tensogram-nb04-"))
nc_path = tmpdir / "demo.nc"
tgm_path = tmpdir / "demo.tgm"

rng = np.random.default_rng(seed=20260404)
ntime, nlat, nlon = 4, 20, 40

with netCDF4.Dataset(nc_path, "w", format="NETCDF4") as ds:
    ds.Conventions = "CF-1.10"
    ds.title = "Tensogram notebook 04 demo"
    ds.institution = "ECMWF / tensogram examples"

    ds.createDimension("time", ntime)
    ds.createDimension("lat", nlat)
    ds.createDimension("lon", nlon)

    lat = ds.createVariable("lat", "f4", ("lat",))
    lat.units = "degrees_north"
    lat.standard_name = "latitude"
    lat.axis = "Y"
    lat[:] = np.linspace(-85, 85, nlat, dtype=np.float32)

    lon = ds.createVariable("lon", "f4", ("lon",))
    lon.units = "degrees_east"
    lon.standard_name = "longitude"
    lon.axis = "X"
    lon[:] = np.linspace(0, 350, nlon, dtype=np.float32)

    time = ds.createVariable("time", "f8", ("time",))
    time.units = "days since 2000-01-01 00:00:00"
    time.calendar = "gregorian"
    time.standard_name = "time"
    time.axis = "T"
    time[:] = np.arange(ntime, dtype=np.float64)

    # Packed temperature: int16 on disk, f64 after unpacking.
    temp = ds.createVariable("temperature", "i2", ("time", "lat", "lon"),
                             fill_value=-32768)
    temp.units = "K"
    temp.standard_name = "air_temperature"
    temp.long_name = "2 metre temperature"
    temp.scale_factor = 0.01
    temp.add_offset = 273.15
    temp.cell_methods = "time: mean"

    # Write unpacked values — netCDF4 applies scale_factor/add_offset
    # automatically on the way out to disk.
    kelvin = rng.uniform(240.0, 310.0, size=(ntime, nlat, nlon))
    temp[:] = kelvin

print(f"wrote {nc_path}  ({nc_path.stat().st_size:,} bytes)")

## 2. Convert: `tensogram.convert_netcdf(...)`

This is the one-liner you'd actually write in production:

In [ ]:
messages = tensogram.convert_netcdf(
    str(nc_path),
    cf=True,                  # lift CF attributes into base[i]["cf"]
    compression="zstd",       # apply zstd across all variables
    compression_level=3,
)

# Materialise as a .tgm file for downstream consumers.
with tgm_path.open("wb") as fh:
    for msg in messages:
        fh.write(msg)

print(f"source   : {nc_path.stat().st_size:>7,} bytes")
print(f"tensogram: {tgm_path.stat().st_size:>7,} bytes  "
      f"({tgm_path.stat().st_size/nc_path.stat().st_size*100:5.1f}% of source)")
print(f"{len(messages)} tensogram message(s)")

## 3. Inspect the output

Each NetCDF variable became a data object with its name preserved under `base[i]["name"]`, and the CF attributes (the subset Tensogram whitelists) are in `base[i]["cf"]`.

In [ ]:
with tensogram.TensogramFile.open(str(tgm_path)) as f:
    print(f"message count: {len(f)}")
    for msg in f:
        print(f"\nmessage: {len(msg.objects)} object(s)")
        for i, (desc, arr) in enumerate(msg.objects):
            entry = msg.metadata.base[i] if i < len(msg.metadata.base) else {}
            name = entry.get("name", "?")
            cf_attrs = sorted(entry.get("cf", {}).keys()) if "cf" in entry else []
            shape_str = str(list(desc.shape))
            print(f"  [{i}] name={name:<12} shape={shape_str:<20} "
                  f"dtype={desc.dtype:<10} compression={desc.compression}")
            if cf_attrs:
                print(f"      cf attrs: {cf_attrs}")

## 4. Cross-verify: Tensogram decode vs direct NetCDF read

The Tensogram converter unpacks `scale_factor` / `add_offset` at read time, producing float64. If we open the source NetCDF directly with `netCDF4` (which also auto-unpacks), the two must agree bit-for-bit.

In [ ]:
# Read the temperature variable from the Tensogram output.
_meta, objects = tensogram.decode(messages[0])
temp_from_tgm = None
for i, (desc, arr) in enumerate(objects):
    if _meta.base[i].get("name") == "temperature":
        temp_from_tgm = arr
        break
assert temp_from_tgm is not None

# Read the temperature variable directly from the source NetCDF.
with netCDF4.Dataset(nc_path, "r") as ds:
    temp_from_nc = ds.variables["temperature"][:].astype(np.float64)

print(f"tgm:   shape={temp_from_tgm.shape} dtype={temp_from_tgm.dtype} "
      f"min={temp_from_tgm.min():.4f} max={temp_from_tgm.max():.4f}")
print(f"nc:    shape={temp_from_nc.shape}  dtype={temp_from_nc.dtype}  "
      f"min={temp_from_nc.min():.4f} max={temp_from_nc.max():.4f}")

# netCDF4 returns a MaskedArray when fill_value is set. Compare data only.
np.testing.assert_array_equal(temp_from_tgm, np.asarray(temp_from_nc))
print("\nTensogram and NetCDF produce identical float64 values.")

## 5. `split_by="variable"` — one message per variable

Some workflows want each variable in its own Tensogram message (e.g. distinct storage keys, or streaming one variable at a time). Use `split_by="variable"` for that.

In [ ]:
per_variable = tensogram.convert_netcdf(str(nc_path), split_by="variable",
                                         cf=True, compression="zstd")

print(f"{len(per_variable)} message(s) — one per variable\n")
for i, msg in enumerate(per_variable):
    meta = tensogram.decode_metadata(msg)
    name = meta.base[0].get("name", "?") if meta.base else "?"
    print(f"  message[{i}]: {name:<12} ({len(msg):,} bytes)")

## 6. xarray integration — `engine="tensogram"`

The `tensogram-xarray` package registers an xarray backend engine. Once installed, any `.tgm` file opens with `xr.open_dataset(path, engine="tensogram")` and behaves like a native xarray Dataset, including `.sel()`, `.isel()`, lazy range decoding, and dask chunking.

In [ ]:
if not _HAS_XARRAY:
    print("skipped: tensogram_xarray is not installed")
else:
    ds = xr.open_dataset(
        str(tgm_path),
        engine="tensogram",
        variable_key="name",   # use base[i]["name"] as the xarray variable name
    )
    print(ds)

In [ ]:
if _HAS_XARRAY:
    # Pick a time slice and plot it. The xarray backend auto-detects
    # the dimension names from CF metadata when available.
    temperature_var = ds["temperature"]
    time_dim = temperature_var.dims[0]  # first dim, whatever it's called
    snapshot = temperature_var.isel({time_dim: 0})
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(snapshot.values - 273.15, origin="lower", cmap="RdYlBu_r", aspect="auto")
    ax.set_title("temperature[time=0] — decoded lazily via the xarray backend")
    plt.colorbar(im, ax=ax, label="°C")
    plt.tight_layout()
    plt.show()
    ds.close()
else:
    print("skipped: no xarray backend")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"removed {tmpdir}")

## Where to go next

- [`05_validation_and_parallelism.ipynb`](05_validation_and_parallelism.ipynb) — validate your `.tgm` files and scale encoding across threads.
- [`../../docs/src/guide/xarray-integration.md`](../../docs/src/guide/xarray-integration.md) — full xarray-backend reference.
- [`../../docs/src/grib/netcdf-metadata.md`](../../docs/src/grib/netcdf-metadata.md) — the NetCDF → Tensogram metadata mapping reference.